In [1]:
%matplotlib inline
import random
import torch
import inspect


In [2]:
class HyperParameters:
    def save_hyperparameters(self, ignore=[]):
        frame = inspect.currentframe().f_back
        _,_,_,local_vars=inspect.getargvalues(frame)
        self.hparams={k:v for k,v in local_vars.items()
                      if k not in set(ignore+['self']) and not k.startswith('_')}
        for k,v in self.hparams.items():
            setattr(self,k,v)

class DataModule(HyperParameters):
    def __init__(self, root='../data', num_workers=4):
        self.save_hyperparameters()

    def get_dataloader(self, train):
        raise NotImplementedError

    def train_dataloader(self):
        return self.get_dataloader(train=True)

    def val_dataloader(self):
        return self.get_dataloader(train=False)


In [7]:
class SyntheticRegressionData(DataModule):
    def __init__(self, w, b, noise=0.01, num_train=1000, num_val=1000, batch_size=32):
        super().__init__()
        self.save_hyperparameters()
        n = num_train + num_val
        self.X = torch.randn(n, len(w))
        noise = torch.randn(n, 1)*noise
        self.y = torch.matmul(self.X, w.reshape((-1,1))) + b + noise

data = SyntheticRegressionData(w=torch.tensor([2, -3.4]), b=4.2)
print('feature:', data.X)
print('target:', data.y)
print(len(data.X) == len(data.y))



feature: tensor([[ 1.0654,  1.8469],
        [ 0.4015,  1.4172],
        [ 1.2086,  0.3005],
        ...,
        [-1.5766, -0.2028],
        [-0.1609,  1.2640],
        [ 1.2490, -0.1693]])
target: tensor([[ 0.0671],
        [ 0.1790],
        [ 5.5978],
        ...,
        [ 1.7423],
        [-0.4293],
        [ 7.2756]])
True
